In [29]:
from vistiq.io import ImageWriterConfig, ImageWriter, ImageLoader, ImageLoaderConfig, unstack_image
from vistiq.utils import ArrayIteratorConfig, check_device, resolve_futures 
from vistiq.core import Tiler, TilerConfig, Untiler, UntilerConfig
from vistiq.preprocess import FuncProcessor, FuncProcessorConfig, PreprocessFlow, PreprocessFlowConfig, ResizeConfig, Resize, RescaleConfig, Rescale, DoG, DoGConfig, PreprocessorConfig, Preprocessor
from vistiq.segment import RegionFilterConfig, RegionFilter, RangeFilterConfig, RangeFilter, RegionAnalyzerConfig, RegionAnalyzer 
from vistiq.segment import MicroSAMSegmenter, MicroSAMSegmenterConfig, MicroSAMMerger, MicroSAMMergerConfig
from vistiq.segment import TiledSegmentationFlow, TiledSegmentationFlowConfig, SegmentationFlow, SegmentationFlowConfig
from vistiq.analysis import CoincidenceDetectorConfig, CoincidenceDetector, AnalysisFlowConfig, AnalysisFlow

from prefect import flow, task
from prefect.task_runners import ProcessPoolTaskRunner
from prefect.futures import wait
from prefect.futures import resolve_futures_to_results

import stackview
import os
import copy
import numpy as np
import math
import logging

# Configure logger and check availability of accelerators

In [30]:
import vistiq
logger = logging.getLogger(vistiq.__name__)

logger.info(f"Available Torch accelerators: {check_device()}")

2026-06-10 23:48:20,364 - INFO - Found mps device: Apple Metal (MPS)
2026-06-10 23:48:20,365 - INFO - Available Torch accelerators: mps


# Load image

In [31]:
#path="/standard/vol191/siegristlab/Microsam_Segmentation/Conditional Split/control_24+48/DCP1/1_Dpn.tif"
#path="/standard/vol191/siegristlab/Microsam_Segmentation/24h/AkhGal4 x OR Susie/Scrib488 Dpn555 EdU 647/Raw files/Animal 1.lif"
path="Animal 1.lif"
#path="/Users/khs3z/Documents/SDS_/projects/Siegrist/Microsam_Segmentation/24h/AkhGal4 x OR Susie/Scrib488 Dpn555 EdU 647/Animal 1.lif"

scene_index = 0

embedding_path = "./embeddings"
#embedding_path = "/standard/vol191/siegristlab/Sagar/microsam/embeddings/"

In [32]:
ilc = ImageLoaderConfig(
    squeeze=True, 
    rename_channel={"Red": "Dpn", "Green": "Scrib", "Blue": "EdU"}, 
    scene_index=scene_index, 
    split_channels=False,
    #substack="C:3"
)
img, metadata = ImageLoader(ilc).run(path)

2026-06-10 23:48:21,990 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-06-10 23:48:22,057 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-06-10 23:48:22,067 - INFO - Loading image from: Animal 1.lif
2026-06-10 23:48:22,204 - INFO - Scenes found: ('Series001', 'Series002', 'Series003')
2026-06-10 23:48:22,317 - INFO - Loaded image: Animal 1.lif scene=0 -> shape=(3, 93, 512, 512) dtype=uint8, channel_names=['Scrib', 'EdU', 'Dpn']
2026-06-10 23:48:22,319 - INFO - Loaded image with shape: (3, 93, 512, 512), dtype: uint8
2026-06-10 23:48:22,320 - INFO - Finished in state Completed()


In [33]:
if "C" in metadata["axes"]:
    vimg = np.concatenate(np.unstack(img, axis=0), axis=-1)
else:
    vimg = img
stackview.slice(vimg)
#stackview.switch(img, colormap=["pure_green", "pure_blue", "pure_red"], toggleable=True)

# Preprocess

In [34]:
ppcfg = PreprocessFlowConfig(
    processors = [
        RescaleConfig(
            low=2, 
            high=98, 
            dtype=np.uint8, 
            iterator_config=ArrayIteratorConfig(slice_def=(-3,-2,-1)) # ZYX over each channel
        ),
        FuncProcessorConfig(
            func="skimage.filters.gaussian",
            kwargs={"sigma": 1.0},
            iterator_config=ArrayIteratorConfig(slice_def=(-2,-1)) # YX over each Z-plane and channel
        ),
        FuncProcessorConfig(
            func="skimage.exposure.adjust_gamma",
            kwargs={"gamma": 0.2},
            iterator_config=ArrayIteratorConfig(slice_def=(-3,-2,-1)) # ZYX over each channel
        ),
        FuncProcessorConfig(
            func="skimage.exposure.adjust_sigmoid",
            iterator_config=ArrayIteratorConfig(slice_def=(-3,-2,-1)) # ZYX over each channel
        ),
        RescaleConfig(
            dtype=np.uint8, 
            iterator_config=ArrayIteratorConfig(slice_def=(-3,-2,-1)) # ZYX over each channel
        ),
        FuncProcessorConfig(
            func="numpy.max", 
            kwargs={"axis":("C")}, # Project all channels into one
            strict_axis=False,     # don't throw exception if the input is a single channel image already
            dtype=np.uint16,
        ),
    ]
)
c_img, c_metadata = PreprocessFlow(ppcfg).run(img, metadata=metadata, workers=-1)
metadata, c_metadata

2026-06-10 23:48:23,738 - INFO - Setting up PreprocessFlow with Rescale,FuncProcessor,FuncProcessor,FuncProcessor,Rescale,FuncProcessor
2026-06-10 23:48:23,883 - INFO - HTTP Request: POST https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/flows/ "HTTP/1.1 200 OK"
2026-06-10 23:48:24,186 - INFO - HTTP Request: POST https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/flow_runs/ "HTTP/1.1 201 Created"
2026-06-10 23:48:24,294 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-06-10 23:48:24,495 - INFO - HTTP Request: PATCH https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/flow_runs/06a2a300-81b2-7df1-8000-53c3493b90b0 "HTTP/1.1 204 N

({'scene_index': 0,
  'dim_order': 'CZYX',
  'axes': ['C', 'Z', 'Y', 'X'],
  'channel_names': ['Scrib', 'EdU', 'Dpn'],
  'channel_axis': 0,
  'shape': (3, 93, 512, 512),
  'dims': <Dimensions [C: 3, Z: 93, Y: 512, X: 512]>,
  'pixel_unit': 'um',
  'scale': Scale(T=None, C=None, Z=-0.9999284782608696, Y=0.3000933463796477, X=0.3000933463796477),
  'physical_pixel_sizes': PhysicalPixelSizes(Z=-0.9999284782608696, Y=0.3000933463796477, X=0.3000933463796477)},
 {'scene_index': 0,
  'dim_order': 'ZYX',
  'axes': ['Z', 'Y', 'X'],
  'channel_names': ['Scrib', 'EdU', 'Dpn'],
  'channel_axis': 0,
  'shape': (93, 512, 512),
  'dims': <Dimensions [Z: 93, Y: 512, X: 512]>,
  'pixel_unit': 'um',
  'scale': Scale(T=None, C=None, Z=-0.9999284782608696, Y=0.3000933463796477, X=0.3000933463796477),
  'physical_pixel_sizes': PhysicalPixelSizes(Z=-0.9999284782608696, Y=0.3000933463796477, X=0.3000933463796477)})

In [35]:
stackview.slice(c_img)

# Segment Lobes 3D

In [36]:
mscfg = MicroSAMSegmenterConfig(
    iterator_config=ArrayIteratorConfig(slice_def=()),
    embedding_path=embedding_path,
)

rfcfg = RegionFilterConfig(
    filters=[
        RangeFilterConfig(
            attribute="cross_sectional_area-xy", 
            range=(2000, np.inf)
        ),
        RangeFilterConfig(
            attribute="cross_sectional_area-xz", 
            range=(2000, np.inf)
        ),
        RangeFilterConfig(
            attribute="cross_sectional_area-yz", 
            range=(2000, np.inf)
        ),
        RangeFilterConfig(
            attribute="aspect_ratio", 
            range=(0.5, 1.0)
        ),
    ]
)

tsfcfg = TiledSegmentationFlowConfig(
    segmenter = mscfg,
    region_filter = rfcfg,
    tile_factor=(3,3),
    resize_factor=(0.25, 0.25),
    iou_threshold=0.5,
    consensus_threshold=0.75,
)
lobe_labels = TiledSegmentationFlow(tsfcfg).run(c_img, metadata=c_metadata, workers=2, verbose=0)

2026-06-10 23:48:33,211 - INFO - HTTP Request: POST https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/flows/ "HTTP/1.1 200 OK"
2026-06-10 23:48:33,404 - INFO - HTTP Request: POST https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/flow_runs/ "HTTP/1.1 201 Created"
2026-06-10 23:48:33,471 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-06-10 23:48:33,587 - INFO - HTTP Request: PATCH https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/flow_runs/06a2a301-1605-7cb6-8000-75d530e1237e "HTTP/1.1 204 No Content"
2026-06-10 23:48:33,730 - INFO - HTTP Request: POST https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c

Using apple MPS device.


2026-06-10 23:48:39,637 - INFO - Running MicroSAMSegmenter with config: classname='Configurable' package='vistiq.core' version=None command_group=None iterator_config=ArrayIteratorConfig(classname=None, slice_def=()) batch_size=10 preferred_backend='processes' tile_shape=None output_type='stack' output_shape=None output_axes=None recompute_scale=False squeeze=True split_axis=None split_channels=False rename_channel=None model_type='vit_l_lm' checkpoint=None embedding_path='./embeddings' pred_iou_thresh=0.88 stability_score_thresh=0.95 box_nms_thresh=0.7 crop_nms_thresh=0.7 min_mask_region_area=0 output_mode='instance_segmentation' with_background=True device=None device_no=0 gpu_fraction=1.0
2026-06-10 23:48:39,638 - INFO - StackProcessor.run: received workers=2 (type: <class 'int'>)
2026-06-10 23:48:39,638 - INFO - Found mps device: Apple Metal (MPS)
2026-06-10 23:48:39,638 - INFO - Set MPS memory fraction to 1.0
2026-06-10 23:48:39,682 - INFO - Using ./embeddings/47b0a949323f1bcee23a

DEBUG: entered LabelRemover.run
DEBUG: region_properties type = <class 'list'>


2026-06-10 23:48:59,373 - INFO - Finished in state Completed()
2026-06-10 23:48:59,375 - INFO - Finished in state Completed()
2026-06-10 23:48:59,784 - INFO - Running Untiler with config: classname='Configurable' package='vistiq.core' version=None command_group=None iterator_config=ArrayIteratorConfig(classname=None, slice_def=()) batch_size=10 preferred_backend='threads' tile_shape=None output_type='stack' output_shape=None output_axes=None recompute_scale=False squeeze=True split_axis=None split_channels=False rename_channel=None factor=(3, 3)
2026-06-10 23:48:59,784 - INFO - StackProcessor.run: received workers=-1 (type: <class 'int'>)
2026-06-10 23:48:59,862 - INFO - factor=(3, 3), array.shape=(18, 93, 399, 399), vs.shape=(3, 18, 93, 399, 133), hs.shape=(3, 3, 18, 93, 133, 133), untiled.shape=(9, 18, 93, 133, 133)
2026-06-10 23:48:59,865 - INFO - Finished in state Completed()
2026-06-10 23:49:00,115 - INFO - Found mps device: Apple Metal (MPS)
2026-06-10 23:49:00,176 - INFO - Finis

DEBUG: entered LabelRemover.run
DEBUG: region_properties type = <class 'list'>


2026-06-10 23:49:01,558 - INFO - HTTP Request: POST https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/flow_runs/06a2a301-1605-7cb6-8000-75d530e1237e/set_state "HTTP/1.1 201 Created"
2026-06-10 23:49:02,015 - INFO - Finished in state Completed()


In [37]:
print (f"Unique labels (incl. background): {np.unique(lobe_labels)}")

Unique labels (incl. background): [0 1 2]


In [38]:
vlabels = np.concatenate(len(metadata["channel_names"])*[lobe_labels], axis=-1)
stackview.blend(vimg.astype("uint16"), vlabels.astype("uint64"), blend_factor=40)

# Analyze regions

In [39]:
racfg = RegionAnalyzerConfig(
    properties=["slice_annotations","volume", "centroid", "cross_sectional_area", "bbox", "aspect_ratio"],
    iterator_config = ArrayIteratorConfig(slice_def=()),
    output_type="dataframe",
    map_axes=True,
)
ra = RegionAnalyzer(racfg)

lobe_measurements = ra.run(lobe_labels, metadata=c_metadata)

2026-06-10 23:49:02,318 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-06-10 23:49:02,362 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-06-10 23:49:02,599 - INFO - Running RegionAnalyzer with config: classname='Configurable' package='vistiq.core' version=None command_group=None iterator_config=ArrayIteratorConfig(classname=None, slice_def=()) batch_size=10 preferred_backend='threads' tile_shape=None output_type='dataframe' output_shape=None output_axes=None recompute_scale=False squeeze=True split_axis=None split_channels=False rename_channel=None properties=['label', 'stack_id', 'slice_id', 'object_id', 'slice_annotations', 'centroid', 'cross_sectional_area', 'bbox', 'aspect_ratio', 'area'] ma

In [40]:
lobe_measurements.sort_values(["volume"],ascending=False)

,centroid-z,centroid-y,centroid-x,bbox-start-z,bbox-start-y,bbox-start-x,bbox-end-z,bbox-end-y,bbox-end-x,volume,aspect_ratio-yz,aspect_ratio-xz,aspect_ratio-xy,aspect_ratio,cross_sectional_area-yz,cross_sectional_area-xz,cross_sectional_area-xy,object_id,stack_id,slice_id
label,,,,,,,,,,,,,,,,,,,,
1,-51.924296,114.406300,93.833818,21,228,132,89,504,468,297263.014140,0.831820,0.780466,0.747684,0.690988,4312.633105,5623.347091,5185.785657,e7b94c6614a444d2856d9a35a7682e63,0ef04c28cd124421a438a3e71c55bd68,d7a03d00bda746e38e4ec357b94fd882
2,-52.975655,43.312153,58.029617,22,0,48,91,324,336,286602.585185,0.870039,0.826183,0.736390,0.680723,4838.359044,4723.131441,5191.549242,24e7104422f04d26ae574f2d8096195a,0ef04c28cd124421a438a3e71c55bd68,d7a03d00bda746e38e4ec357b94fd882


# Labels to lobe masks

In [41]:
from vistiq.core import labels_to_masks

lobe_masks = labels_to_masks(lobe_labels)
brain_label = (lobe_labels>0).astype("uint16")

# Save label

In [43]:
c_metadata["channel_names"] = ["Lobe"]

b_metadata = copy.deepcopy(c_metadata)
b_metadata["channel_names"] = ["Brain"]

In [44]:
imc = ImageWriterConfig(overwrite=True)
outpath = ".".join(path.split(".")[:-1]) + f"-scene-{scene_index}.tif"
ImageWriter(imc).run(lobe_labels, outpath, metadata=c_metadata)

imc = ImageWriterConfig(overwrite=True)
outpath = ".".join(path.split(".")[:-1]) + f"-scene-{scene_index}.tif"
ImageWriter(imc).run(brain_label, outpath, metadata=b_metadata)


2026-06-10 23:49:20,000 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-06-10 23:49:20,049 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-06-10 23:49:20,188 - INFO - ImageWriter: using config: classname='Configurable' package='vistiq.core' version=None command_group=None path='.' format='tif' overwrite=True writer=None split_channels=False extension='tif'
2026-06-10 23:49:20,188 - INFO - Preparing to save image with metadata: {'scene_index': 0, 'dim_order': 'ZYX', 'axes': ['Z', 'Y', 'X'], 'channel_names': ['Lobe'], 'channel_axis': 0, 'shape': (93, 512, 512), 'dims': <Dimensions [Z: 93, Y: 512, X: 512]>, 'pixel_unit': 'um', 'scale': Scale(T=None, C=None, Z=-0.9999284782608696, Y=0.3000933463796477

# Segment Cells

In [45]:
from typing import Any, List, Tuple
from prefect import task, flow

In [ ]:
@task(name="unwrap")
def unwrap_results(futures) -> tuple[Any,Any]:
    #print(len(futures), type(futures))
    a, b, = wait(futures)#.result()
    return a, b

@task
def unzip_results(results: List[Tuple]):
    # Converts [(1, 11), (2, 12)] into ([1, 2], [11, 12])
    return tuple(zip(*results))

In [50]:
# specify preprocessing config 
ppcfg = PreprocessFlowConfig(
    processors = [
        #DoGConfig(
        #    sigma_low=1, # 5, 
        #    sigma_high=2, #12, 
        #    normalize=True,
        #    iterator_config=ArrayIteratorConfig(slice_def=(-2,-1)) # YX over each Z focal plane and channel
        #)
    ]
)


In [51]:
# Specify segmentation config
mscfg = MicroSAMSegmenterConfig(
    iterator_config=ArrayIteratorConfig(slice_def=()),
    embedding_path=embedding_path,
    #gpu_fraction=0.3,
)

min_cell_radius = 2.0
max_cell_radius = 7.0
rfcfg = RegionFilterConfig(
    filters=[
        RangeFilterConfig(
            attribute="cross_sectional_area-xy", 
            range=(np.pi*min_cell_radius**2, np.pi*max_cell_radius**2)
        )
    ]
)

sfcfg = SegmentationFlowConfig(
    segmenter = mscfg,
    region_filter = rfcfg,
)

In [52]:
import pandas as pd
import itertools

@flow
def analyze_cells(labels: list[np.ndarray], metadata: list[dict[str, Any]]) -> list[pd.DataFrame]:
    print ([l.shape for l in labels])
    print ([m["channel_names"] for m in metadata])
    racfg = RegionAnalyzerConfig(
        properties=["volume", "centroid", "cross_sectional_area", "bbox", "aspect_ratio"],
        iterator_config = ArrayIteratorConfig(slice_def=()),
        output_type="dataframe"
    )
    ra = RegionAnalyzer(racfg)

    measurements = ra.run.map(labels, metadata=metadata)

    cdcfg = CoincidenceDetectorConfig(
        method="ios",
        iterator_config=ArrayIteratorConfig(slice_def=()),
        mode="outline",
    )
    label_index_combinations = list(itertools.combinations(range(len(labels)), 2))
    l1 = [labels[c[0]] for c in label_index_combinations]
    l2 = [labels[c[1]] for c in label_index_combinations]
    sn = [(metadata[c[0]]["channel_names"][0], metadata[c[1]]["channel_names"][0]) for c in label_index_combinations]
    print (sn)
    #for la1, la2, sna in zip(l1,l2,sn): 
    cim = CoincidenceDetector(cdcfg).run.map(l1, l2, stack_names=sn)
    return measurements
 

In [65]:
acfg = AnalysisFlowConfig(
    region_analyzer = RegionAnalyzerConfig(
        properties=["volume", "centroid", "cross_sectional_area", "bbox", "aspect_ratio"],
        iterator_config = ArrayIteratorConfig(slice_def=()),
        output_type="dataframe"
    ),
    coincidence_detector = CoincidenceDetectorConfig(
        method="ios",
        iterator_config=ArrayIteratorConfig(slice_def=()),
        mode="outline",
    ),
)

In [66]:
@flow
def full_pipeline(img, metadata=metadata):
    # preprocess
    preprocessed, preprocessed_metadata = PreprocessFlow(ppcfg).run(img, metadata=metadata)
    # split channels
    channels, channel_metadata = unstack_image(preprocessed, preprocessed_metadata, axis=metadata["channel_axis"], strict=False)
    # segment each channel separately
    cell_labels = SegmentationFlow(sfcfg).mapped_run(channels, metadata=channel_metadata)
    # analyze regions in each channel separately
    measurements = AnalysisFlow(acfg).run(cell_labels, metadata=channel_metadata)
    # measurements = analyze_cells([*cell_labels, lobe_labels, brain_label], metadata=[*channel_metadata, c_metadata, b_metadata])
    # make sure to resolve the futures to results
    return (resolve_futures(cell_labels), 
    resolve_futures(channel_metadata), 
    resolve_futures(measurements), 
    resolve_futures(preprocessed), 
    resolve_futures(channels))

In [67]:
cell_labels, cell_metadata, cell_measurements, preprocessed, channels = full_pipeline(img, metadata=metadata)

2026-06-11 00:07:00,462 - INFO - HTTP Request: POST https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/flows/ "HTTP/1.1 200 OK"
2026-06-11 00:07:00,680 - INFO - HTTP Request: POST https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/flow_runs/ "HTTP/1.1 201 Created"
2026-06-11 00:07:00,987 - INFO - HTTP Request: POST https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/flow_runs/06a2a346-4982-78bd-8000-8c908c21510d/set_state "HTTP/1.1 201 Created"
2026-06-11 00:07:01,162 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/flow_runs/06a2a346-4982-78bd-8000-8c908c21510d "HTTP/1.1 200 OK"
2026-06-11 00:07:01,224 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe0

Using apple MPS device.
Using apple MPS device.
Using apple MPS device.


2026-06-11 00:07:15,871 - INFO - Running MicroSAMSegmenter with config: classname='Configurable' package='vistiq.core' version=None command_group=None iterator_config=ArrayIteratorConfig(classname=None, slice_def=()) batch_size=10 preferred_backend='processes' tile_shape=None output_type='stack' output_shape=None output_axes=None recompute_scale=False squeeze=True split_axis=None split_channels=False rename_channel=None model_type='vit_l_lm' checkpoint=None embedding_path='./embeddings' pred_iou_thresh=0.88 stability_score_thresh=0.95 box_nms_thresh=0.7 crop_nms_thresh=0.7 min_mask_region_area=0 output_mode='instance_segmentation' with_background=True device=None device_no=0 gpu_fraction=1.0
2026-06-11 00:07:15,872 - INFO - StackProcessor.run: received workers=-1 (type: <class 'int'>)
2026-06-11 00:07:15,873 - INFO - Found mps device: Apple Metal (MPS)
2026-06-11 00:07:15,873 - INFO - Set MPS memory fraction to 1.0
2026-06-11 00:07:15,882 - INFO - Using ./embeddings/040b65c384168bc8f1a

DEBUG: entered LabelRemover.run
DEBUG: region_properties type = <class 'list'>


2026-06-11 00:08:05,156 - INFO - Finished in state Completed()
2026-06-11 00:08:05,225 - INFO - Finished in state Completed()
2026-06-11 00:08:05,226 - INFO - Finished in state Completed()
2026-06-11 00:08:05,284 - INFO - Finished in state Completed()
2026-06-11 00:08:05,330 - INFO - Finished in state Completed()
2026-06-11 00:08:05,492 - INFO - DEBUG: entered Relabeler.run
2026-06-11 00:08:06,925 - INFO - Finished in state Completed()
2026-06-11 00:08:06,960 - INFO - Finished in state Completed()
2026-06-11 00:08:07,113 - INFO - DEBUG: entered Relabeler.run
2026-06-11 00:08:10,130 - INFO - Finished in state Completed()
2026-06-11 00:08:10,174 - INFO - Creating RegionAnalyzer for region filter with properties: ['label', 'object_id', 'slice_id', 'stack_id', 'centroid', 'cross_sectional_area-xy']
2026-06-11 00:08:10,413 - INFO - Running RegionAnalyzer with config: classname='Configurable' package='vistiq.core' version=None command_group=None iterator_config=ArrayIteratorConfig(classname=

DEBUG: entered LabelRemover.run
DEBUG: region_properties type = <class 'list'>


2026-06-11 00:08:11,280 - INFO - Finished in state Completed()
2026-06-11 00:08:11,346 - INFO - Finished in state Completed()
2026-06-11 00:08:11,348 - INFO - Finished in state Completed()
2026-06-11 00:08:13,055 - INFO - Finished in state Completed()
2026-06-11 00:08:13,087 - INFO - Creating RegionAnalyzer for region filter with properties: ['label', 'object_id', 'slice_id', 'stack_id', 'centroid', 'cross_sectional_area-xy']
2026-06-11 00:08:13,338 - INFO - Running RegionAnalyzer with config: classname='Configurable' package='vistiq.core' version=None command_group=None iterator_config=ArrayIteratorConfig(classname=None, slice_def=()) batch_size=10 preferred_backend='threads' tile_shape=None output_type='list' output_shape=None output_axes=None recompute_scale=False squeeze=True split_axis=None split_channels=False rename_channel=None properties=['label', 'object_id', 'slice_id', 'stack_id', 'centroid', 'cross_sectional_area-xy'] map_axes=False expand_coordinates=False
2026-06-11 00:0

DEBUG: entered LabelRemover.run
DEBUG: region_properties type = <class 'list'>


2026-06-11 00:08:15,138 - INFO - Finished in state Completed()
2026-06-11 00:08:15,207 - INFO - Finished in state Completed()
2026-06-11 00:08:15,209 - INFO - Finished in state Completed()
2026-06-11 00:08:15,485 - INFO - HTTP Request: POST https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/flow_runs/06a2a346-84f6-7c09-8000-5e32615da878/set_state "HTTP/1.1 201 Created"
2026-06-11 00:08:15,554 - INFO - Finished in state Completed('All states completed.')
2026-06-11 00:08:15,783 - INFO - HTTP Request: POST https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/task_runs/ "HTTP/1.1 201 Created"
2026-06-11 00:08:15,914 - INFO - HTTP Request: POST https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/flow_runs/filter "HTTP/1.1 200 OK"
2026-06-11 00:08:16,020 - INFO - HTTP Request: P

In [61]:
measurements = analyze_cells([*cell_labels, lobe_labels, brain_label], metadata=[*cell_metadata, c_metadata, b_metadata])

2026-06-11 00:02:38,228 - INFO - HTTP Request: POST https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/flows/ "HTTP/1.1 200 OK"
2026-06-11 00:02:38,536 - INFO - HTTP Request: POST https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/flow_runs/ "HTTP/1.1 201 Created"
2026-06-11 00:02:38,841 - INFO - HTTP Request: POST https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/flow_runs/06a2a335-e6af-71c0-8000-03bec896c4be/set_state "HTTP/1.1 201 Created"
2026-06-11 00:02:38,931 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/flow_runs/06a2a335-e6af-71c0-8000-03bec896c4be "HTTP/1.1 200 OK"
2026-06-11 00:02:39,020 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe0

[(93, 512, 512), (93, 512, 512), (93, 512, 512), (93, 512, 512), (93, 512, 512)]
[['Scrib'], ['EdU'], ['Dpn'], ['Lobe'], ['Brain']]
[('Scrib', 'EdU'), ('Scrib', 'Dpn'), ('Scrib', 'Lobe'), ('Scrib', 'Brain'), ('EdU', 'Dpn'), ('EdU', 'Lobe'), ('EdU', 'Brain'), ('Dpn', 'Lobe'), ('Dpn', 'Brain'), ('Lobe', 'Brain')]


2026-06-11 00:02:39,348 - INFO - Running RegionAnalyzer with config: classname='Configurable' package='vistiq.core' version=None command_group=None iterator_config=ArrayIteratorConfig(classname=None, slice_def=()) batch_size=10 preferred_backend='threads' tile_shape=None output_type='dataframe' output_shape=None output_axes=None recompute_scale=False squeeze=True split_axis=None split_channels=False rename_channel=None properties=['label', 'stack_id', 'slice_id', 'object_id', 'centroid', 'cross_sectional_area', 'bbox', 'aspect_ratio', 'area'] map_axes=False expand_coordinates=False
2026-06-11 00:02:39,349 - INFO - StackProcessor.run: received workers=-1 (type: <class 'int'>)
2026-06-11 00:02:39,365 - INFO - RegionAnalyzer: Applying scale: (-0.9999284782608696, 0.3000933463796477, 0.3000933463796477), labels.shape=(93, 512, 512), channel_names=['Brain']
2026-06-11 00:02:39,432 - INFO - Running RegionAnalyzer with config: classname='Configurable' package='vistiq.core' version=None comman

In [62]:
stackview.slice(np.concatenate(cell_labels, axis=-1))

In [63]:
print (type(cell_measurements))
for k,v in cell_measurements.items():
    if isinstance(v, pd.DataFrame):
        print (k, v.head())
    else:
        print (f"{k} *** {v[0][:3]} *** {v[1].keys()} *** {v[1].values()}")

<class 'dict'>
coincidence: Scrib vs EdU *** [{'Scrib': 1, 'EdU': 1, 'score': 0.0, 'above_threshold': False}, {'Scrib': 1, 'EdU': 2, 'score': 0.0, 'above_threshold': False}, {'Scrib': 1, 'EdU': 3, 'score': 0.0, 'above_threshold': False}] *** dict_keys(['Scrib', 'EdU']) *** dict_values([       EdU +  ios EdU +
label                  
1      False        0.0
2      False        0.0
3      False        0.0
4      False        0.0
5      False        0.0
...      ...        ...
453    False        0.0
454    False        0.0
455    False        0.0
456    False        0.0
457    False        0.0

[457 rows x 2 columns],        Scrib +  ios Scrib +
label                      
1        False      0.00000
2        False      0.00000
3        False      0.00000
4        False      0.00000
5        False      0.00000
...        ...          ...
655      False      0.00000
656      False      0.40184
657      False      0.00000
658      False      0.00000
659      False      0.00000

[659 rows x

# Hierarchical label decomposition

In [ ]:
from vistiq.analysis import CoincidenceDetector, CoincidenceDetectorConfig

cdcfg = CoincidenceDetectorConfig(
    method="ios",
    iterator_config=ArrayIteratorConfig(slice_def=()),
    mode="outline",
)
df = CoincidenceDetector(cdcfg).run(lobe_labels, cell_labels[2], stack_names=["Lobe", "Dpn"])   

In [ ]:
df[1].values()

In [64]:
from vistiq.analysis.coincidence import box_iou_batch_3d, labels_iou_batch_3d, labels_iou_batch_3d_torch
from joblib import Parallel, delayed

bbox_cols = [b for b in lobe_measurements.columns.to_list() if "bbox" in b]
lobe_bboxes = lobe_measurements[bbox_cols].to_numpy()
threshold = 0.5
cell_boxes_list = [cm[bbox_cols].to_numpy() for cm in cell_measurements]
#ios = Parallel(verbose=10)(delayed(box_iou_batch_3d)(lobe_bboxes, cb, overlap_metric="IOS") for cb in cell_boxes_list)
ios = Parallel(verbose=10, prefer="threads", n_jobs=1)(delayed(labels_iou_batch_3d_torch)(lobe_labels, cl, overlap_metric="IOS", dense_pair_fraction=0.75) for cl in cell_labels)
#for cl, cm in zip(cell_labels,cell_measurements):
#    cell_bboxes = cm[bbox_cols].to_numpy()
#    #ios = labels_iou_batch_3d(lobe_labels, cl, overlap_metric="IOS")
#    ios = box_iou_batch_3d(lobe_bboxes, cell_bboxes, overlap_metric="IOS")
#    print (ios.shape, len(np.argwhere(ios>threshold)), len(np.argwhere(ios<=threshold)))
for i in ios:
    print (i.shape)

TypeError: string indices must be integers, not 'list'

# View in Napari

In [ ]:
import napari
viewer = napari.Viewer()

In [ ]:
scale = metadata["physical_pixel_sizes"]
channel_colors = ("green", "blue", "red")
nimg = img#np.expand_dims(img, axis=0)

# add brain label
viewer.add_labels(brain, name="Brain", scale=scale)

# add lobe labels
for ch, l, m in zip(metadata["channel_names"],cell_labels, cell_measurements):
    new_m = m.copy().reset_index()
    background = pd.DataFrame({c: [0] if c=="label" else [np.nan] for c in new_m.columns.to_list()})
    new_m = pd.concat([background, new_m], ignore_index=True)
    # print (new_m)
    viewer.add_labels(l, name=f"{ch}-Labels", features=new_m, scale=scale)

# add cell labels for each channel
ch_images, ch_metadata = unstack_image(img, metadata=metadata, axis="C", strict=False)
for name, c_img, color in zip(metadata["channel_names"], ch_images, channel_colors):
    viewer.add_image(c_img, name=f"{name}", scale=scale, colormap=color, blending="additive")
    


In [ ]:
dl1 = viewer.layers["Dpn-Labels-Lobe 1"]
dl2 = viewer.layers["Dpn-Labels-Lobe 2"]
print (np.intersect1d(np.unique(dl1.data), np.unique(dl2.data)))
